<a href="https://colab.research.google.com/github/vuongthingocanh006/HBBA_Competition/blob/main/HBBA_Competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Hãy bắt đầu đọc các file sample nhé.

In [2]:
import pandas as pd
import numpy as np

# 1. Đọc dữ liệu gốc
df = pd.read_csv('train.csv')

# 2. Chuyển đổi định dạng ngày tháng
df['Date'] = pd.to_datetime(df['Date'])

# 3. Thay thế ',' thành '.' và ép kiểu số cho các cột Object tài chính
cols_to_convert = ['UnitPrice', 'Unit Cost', 'Cost Amount']
for col in cols_to_convert:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.'), errors='coerce')

# Ép kiểu cho cột SalesAmount (vốn đang là int) sang float để đồng bộ tính toán
df['SalesAmount'] = df['SalesAmount'].astype(float)

# Gom nhóm theo Ngày và SKU để có tổng lượng bán mỗi ngày (gồm cả ngày bị âm)
daily_df = df.groupby(['Date', 'ItemCode'], as_index=False).agg({
    'Quantity': 'sum',
    'SalesAmount': 'sum',
    'Cost Amount': 'sum',
    'UnitPrice': 'mean',
    'Unit Cost': 'mean'
})

/tmp/ipykernel_4006/4002807662.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('train.csv')


In [3]:
## Xử lý dữ liệu âm (bị trả về)
def back_distribute_returns(group):
    """
    Hàm xử lý khấu trừ ngược lượng hàng trả cho từng SKU cụ thể.
    Sắp xếp ngược thời gian để bù trừ lượng âm vào các ngày bán dương trước đó.
    """
    # Sắp xếp từ ngày mới nhất về ngày cũ nhất
    group = group.sort_values(by='Date', ascending=False).reset_index(drop=True)

    qty = group['Quantity'].values
    sales = group['SalesAmount'].values
    cost = group['Cost Amount'].values

    cum_negative_qty = 0
    cum_negative_sales = 0
    cum_negative_cost = 0

    # Chạy từ ngày mới nhất ngược về quá khứ
    for i in range(len(qty)):
        # Cộng dồn lượng âm (nếu có)
        if qty[i] < 0:
            cum_negative_qty += qty[i]
            cum_negative_sales += sales[i]
            cum_negative_cost += cost[i]

            # Tạm thời đưa ngày bị âm này về 0
            qty[i] = 0
            sales[i] = 0
            cost[i] = 0

        # Nếu đang có lượng âm cần bù trừ và gặp một ngày bán được hàng (qty[i] > 0)
        elif cum_negative_qty < 0 and qty[i] > 0:
            # Ngày dương này đủ để bù hết toàn bộ lượng âm tích lũy
            if qty[i] >= abs(cum_negative_qty):
                qty[i] += cum_negative_qty  # Cộng số âm (tức là trừ bớt đi)
                sales[i] += cum_negative_sales
                cost[i] += cum_negative_cost

                # Reset bộ đếm âm về 0 vì đã bù xong
                cum_negative_qty = 0
                cum_negative_sales = 0
                cum_negative_cost = 0
            else:
                # Ngày dương này chỉ bù được một phần lượng âm tích lũy
                cum_negative_qty += qty[i]
                cum_negative_sales += sales[i]
                cum_negative_cost += cost[i]

                # Ngày này bị trừ sạch về 0
                qty[i] = 0
                sales[i] = 0
                cost[i] = 0

    group['Quantity'] = qty
    group['SalesAmount'] = sales
    group['Cost Amount'] = cost

    # Trả về thứ tự thời gian xuôi như cũ (từ cũ đến mới)
    return group.sort_values(by='Date').reset_index(drop=True)

print("Đang tiến hành khấu trừ lượng hàng trả lại (Back-distribution) cho 15,972 SKUs...")
# Áp dụng hàm xử lý cho từng nhóm ItemCode
daily_cleaned = daily_df.groupby('ItemCode', group_keys=False).apply(back_distribute_returns)
print("Hoàn thành khấu trừ ngược!")

# Kiểm tra lại xem còn dòng nào bị âm không
remaining_negatives = (daily_cleaned['Quantity'] < 0).sum()
print(f"Số lượng dòng còn bị âm sau khi xử lý: {remaining_negatives}")

# Đề phòng trường hợp SKU bị trả hàng ngay đầu chuỗi lịch sử (không có ngày quá khứ để trừ)
# Ta áp dụng thêm một lệnh clip để ép tất cả lượng âm sót lại (nếu có) về 0
daily_cleaned['Quantity'] = daily_cleaned['Quantity'].clip(lower=0)
daily_cleaned['SalesAmount'] = daily_cleaned['SalesAmount'].clip(lower=0)
daily_cleaned['Cost Amount'] = daily_cleaned['Cost Amount'].clip(lower=0)

Đang tiến hành khấu trừ lượng hàng trả lại (Back-distribution) cho 15,972 SKUs...
Hoàn thành khấu trừ ngược!
Số lượng dòng còn bị âm sau khi xử lý: 0


/tmp/ipykernel_4006/3269081695.py:63: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_cleaned = daily_df.groupby('ItemCode', group_keys=False).apply(back_distribute_returns)


In [5]:
import pandas as pd
import numpy as np

# --- BƯỚC 3: TẠO KHUNG LƯỚI THỜI GIAN LIÊN TỤC (GRID) ---
unique_skus = daily_cleaned['ItemCode'].unique()
start_date = daily_cleaned['Date'].min()
end_date = daily_cleaned['Date'].max()
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

grid_index = pd.MultiIndex.from_product([date_range, unique_skus], names=['Date', 'ItemCode'])
grid_df = pd.DataFrame(index=grid_index).reset_index()

# Trộn dữ liệu đã khấu trừ ngược vào Khung lưới
grid_df = pd.merge(grid_df, daily_cleaned, on=['Date', 'ItemCode'], how='left')

# Điền giá trị 0 cho các ngày không có giao dịch
grid_df['Quantity'] = grid_df['Quantity'].fillna(0)
grid_df['SalesAmount'] = grid_df['SalesAmount'].fillna(0)
grid_df['Cost Amount'] = grid_df['Cost Amount'].fillna(0)

# Điền khuyết cho các cột Đơn giá và Giá vốn bằng giá lịch sử gần nhất của chính SKU đó
grid_df = grid_df.sort_values(by=['ItemCode', 'Date']).reset_index(drop=True)
grid_df['UnitPrice'] = grid_df.groupby('ItemCode')['UnitPrice'].ffill().bfill()
grid_df['Unit Cost'] = grid_df.groupby('ItemCode')['Unit Cost'].ffill().bfill()

# --- BƯỚC 4: TỐI ƯU HÓA BỘ NHỚ RAM ---
def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        # Exclude object and datetime columns explicitly
        if col_type != object and col_type.kind != 'M': # 'M' for datetime64[ns]
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    return df

grid_df = reduce_mem_usage(grid_df)
print(f"Khung lưới hoàn chỉnh sau Back-distribution có kích thước: {grid_df.shape}")

Khung lưới hoàn chỉnh sau Back-distribution có kích thước: (28014888, 7)


In [6]:
df.head(10)

,Date,Stt,ItemCode,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount
0,2020-11-17,2000004,SKU-08063,12,242700.0000,2184300.0,123559.1,1482709.0
1,2020-11-17,2000003,SKU-09458,600,131818.1818,79090909.0,110000.0,66000000.0
2,2020-11-18,2000007,SKU-08062,6,230000.0000,940909.0,101000.0,606000.0
3,2020-11-18,2000006,SKU-09458,240,270000.0000,44181818.0,110000.0,26400000.0
4,2020-11-18,2000005,SKU-09458,240,270000.0000,44181818.0,110000.0,26400000.0
5,2020-11-20,2000008,SKU-09458,240,245454.5455,44181818.0,110000.0,26400000.0
6,2020-11-21,2000013,SKU-07688,10,154545.4545,1159091.0,67000.0,670000.0
7,2020-11-21,2000013,SKU-07690,8,516363.6364,3098182.0,252000.0,2016000.0
8,2020-11-21,2000013,SKU-07675,10,135454.5455,1015909.0,67000.0,670000.0
9,2020-11-21,2000013,SKU-07691,4,589090.9091,1767273.0,228000.0,912000.0


In [7]:
# 1. Xem số lượng dòng và cột của file
print(df.shape)

# 2. Xem ngẫu nhiên 5 dòng bất kỳ trong file (Thay vì chỉ xem ở đầu)
print(df.sample(5))

# 3. Xem bảng thống kê toán học (Tổng, Trung bình, Nhỏ nhất, Lớn nhất...) của các cột số
print(df.describe())

(711980, 8)
             Date      Stt   ItemCode  Quantity  UnitPrice  SalesAmount  \
454157 2024-03-16  1696871  SKU-08645         1   620000.0     620000.0   
676539 2025-06-09  1900371  SKU-03896         1   794000.0     794000.0   
206179 2022-12-16  1495678  SKU-04584         1  1290000.0    1290000.0   
301845 2023-06-06  1569929  SKU-14891         2   202000.0     395920.0   
501131 2024-06-13  1736774  SKU-06559         2   291000.0     570360.0   

        Unit Cost  Cost Amount  
454157  468044.98     468045.0  
676539  616156.50     616157.0  
206179  927468.00     927468.0  
301845  139183.24     278366.0  
501131  160381.84     320764.0  
                                Date       Quantity     UnitPrice  \
count                         711980  711980.000000  7.119800e+05   
mean   2023-09-26 14:19:42.612994816       3.437250  5.361727e+05   
min              2020-11-17 00:00:00    -998.000000 -3.001100e+07   
25%              2022-11-01 00:00:00       1.000000  1.450000e+